# Applied AI Midterm — Kaggle: continue final SRGAN 27 → 125 → 150

This notebook covers the Kaggle SRGAN training. The public Dogs-vs-Cats dataset and **your Colab transfer checkpoint dataset (epoch 27)** must both be mounted. You must have write access to the configured Kaggle dataset slug; replace the `niklasohlenroth/...` dataset handle and paths with your own for independent reproduction. The training takes many GPU hours.

Kaggle checkpoint dataset snapshots are read-only. **Between phases you must attach the newly uploaded version and restart/recreate the kernel**, or restore the current checkpoint to a writable `/kaggle/working/project`. Do not overwrite a newer backup with an older mounted snapshot.

## Stage A — Restore the transferred epoch-27 experiment
Rebuild the exact 128×128 JPEGs from source images, load the custom PixelShuffle generator, GroupNorm discriminator and full optimizer checkpoint.

In [ ]:
from pathlib import Path
import shutil
import pandas as pd
import tensorflow as tf

# --------------------------------------------------
# 1. Paths
# --------------------------------------------------

TRANSFER_ROOT = Path(
    "/kaggle/input/datasets/niklasohlenroth/"
    "applied-ai-midterm-srgan-checkpoints/project"
)

WORK_ROOT = Path("/kaggle/working/project")

DOGS_BASE = Path("/kaggle/input/datasets/blourdhuraju")

print("=== PATH CHECK ===")
print("TRANSFER_ROOT exists:", TRANSFER_ROOT.exists())
print("WORK_ROOT exists before copy:", WORK_ROOT.exists())
print("DOGS_BASE exists:", DOGS_BASE.exists())

# --------------------------------------------------
# 2. Find Dogs-vs-Cats dataset root
# --------------------------------------------------

dataset_info_files = list(DOGS_BASE.rglob("dataset_info.csv"))

print("\n=== DOGS DATASET SEARCH ===")
print("dataset_info.csv files found:")
for p in dataset_info_files:
    print(" ", p)

if not dataset_info_files:
    raise FileNotFoundError("Could not find dataset_info.csv for dogs-vs-cats dataset.")

DOGS_DATASET_ROOT = dataset_info_files[0].parent

print("\nSelected DOGS_DATASET_ROOT:")
print(DOGS_DATASET_ROOT)

print("\nSubfolders:")
for item in sorted(DOGS_DATASET_ROOT.iterdir()):
    print(" ", item.name)

# --------------------------------------------------
# 3. Copy transferred project to writable working dir
# --------------------------------------------------

if WORK_ROOT.exists():
    print("\nWorking directory already exists. Skipping copy.")
else:
    print("\nCopying project to /kaggle/working ...")
    shutil.copytree(TRANSFER_ROOT, WORK_ROOT)
    print("Copy completed.")

# --------------------------------------------------
# 4. Verify copied training files
# --------------------------------------------------

resume_dir = WORK_ROOT / "checkpoints" / "srgan" / "groupnorm_v2" / "resume"
archive_dir = WORK_ROOT / "checkpoints" / "srgan" / "groupnorm_v2" / "archive"
best_dir = WORK_ROOT / "checkpoints" / "srgan" / "groupnorm_v2" / "best_generators"

latest_ckpt = tf.train.latest_checkpoint(str(resume_dir))
latest_archive = tf.train.latest_checkpoint(str(archive_dir))

print("\n=== COPIED PROJECT CHECK ===")
print("Latest resume checkpoint:", latest_ckpt)
print("Latest archive checkpoint:", latest_archive)
print("Best generator dir exists:", best_dir.exists())

history_file = WORK_ROOT / "results" / "srgan_training_groupnorm_v2.csv"
print("History file exists:", history_file.exists())

if history_file.exists():
    history = pd.read_csv(history_file)
    print("\nLast history row:")
    print(history.tail(1).to_string(index=False))

print("\nSetup copy check finished.")

In [ ]:

from pathlib import Path
from PIL import Image, ImageOps
import pandas as pd
import hashlib
import os
import time

# ============================================================
# KAGGLE - PREPARE 128x128 RGB IMAGES FOR SRGAN
# ============================================================

PROJECT_DIR = Path("/kaggle/working/project")

DOGS_DATASET_ROOT = Path(
    "/kaggle/input/datasets/blourdhuraju/"
    "dogs-vs-cats-classification/"
    "dogs-vs-cats-classification"
)

PREPARED_DIR = Path("/kaggle/working/prepared_rgb128")
PREPARED_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = (128, 128)
JPEG_QUALITY = 95

# ------------------------------------------------------------
# 1. Load original dataset split
# ------------------------------------------------------------

split_path = PROJECT_DIR / "data" / "dataset_split.csv"

if not split_path.is_file():
    raise FileNotFoundError(
        f"Original dataset split not found: {split_path}"
    )

split_df = pd.read_csv(split_path)

assert len(split_df) == 24930
assert split_df["relative_path"].is_unique
assert set(split_df["label"].unique()) == {0, 1}

expected_counts = {
    "train": 15705,
    "validation": 1746,
    "test": 7479
}

assert split_df["split"].value_counts().to_dict() == expected_counts

# ------------------------------------------------------------
# 2. Rebuild original image paths for Kaggle
# ------------------------------------------------------------

prefix = "dogs-vs-cats-classification/"

def fix_image_path(relative_path):

    path = str(relative_path).replace("\\", "/")

    if path.startswith(prefix):
        path = path[len(prefix):]

    return path

split_df["kaggle_relative_path"] = (
    split_df["relative_path"].apply(fix_image_path)
)

split_df["image_path"] = (
    split_df["kaggle_relative_path"].apply(
        lambda p: str(DOGS_DATASET_ROOT / p)
    )
)

missing_originals = [
    p for p in split_df["image_path"]
    if not Path(p).is_file()
]

if missing_originals:
    raise FileNotFoundError(
        f"{len(missing_originals)} original images missing. "
        f"First: {missing_originals[0]}"
    )

print("=== ORIGINAL DATASET ===")
print("Total images:", len(split_df))
print("Missing images:", len(missing_originals))
print("\nSplit distribution:")
print(split_df["split"].value_counts())

# ------------------------------------------------------------
# 3. Create deterministic prepared-image filenames
# ------------------------------------------------------------

def get_prepared_path(relative_path):

    file_id = hashlib.sha256(
        str(relative_path).encode("utf-8")
    ).hexdigest()[:24]

    return PREPARED_DIR / f"{file_id}.jpg"

prepared_df = split_df.copy()

prepared_df["prepared_path"] = (
    prepared_df["relative_path"].apply(
        lambda p: str(get_prepared_path(p))
    )
)

assert prepared_df["prepared_path"].is_unique

# ------------------------------------------------------------
# 4. Prepare all images
# ------------------------------------------------------------

start_time = time.perf_counter()

created = 0
reused = 0
errors = []

print("\n=== PREPARING SRGAN IMAGES ===")
print("Total images:", len(prepared_df))
print("Target size:", IMAGE_SIZE)
print("Format: RGB JPEG")
print("Resizing: Bicubic")
print()

for i, row in enumerate(
    prepared_df.itertuples(index=False),
    start=1
):

    src = Path(row.image_path)
    dst = Path(row.prepared_path)

    if dst.is_file() and dst.stat().st_size > 0:
        reused += 1

    else:
        tmp = dst.with_suffix(".part")

        try:

            with Image.open(src) as img:

                img = ImageOps.exif_transpose(img)

                img = img.convert("RGB")

                img = img.resize(
                    IMAGE_SIZE,
                    resample=Image.Resampling.BICUBIC
                )

                img.save(
                    tmp,
                    format="JPEG",
                    quality=JPEG_QUALITY
                )

            os.replace(tmp, dst)

            created += 1

        except Exception as exc:

            errors.append(
                (str(src), str(exc))
            )

            tmp.unlink(missing_ok=True)

    if i % 2500 == 0 or i == len(prepared_df):

        print(
            f"Processed {i:,}/{len(prepared_df):,} | "
            f"New: {created:,} | "
            f"Reused: {reused:,} | "
            f"Errors: {len(errors)}"
        )

# ------------------------------------------------------------
# 5. Verify prepared images
# ------------------------------------------------------------

missing_prepared = [
    p for p in prepared_df["prepared_path"]
    if not Path(p).is_file()
]

if errors or missing_prepared:

    print("\nFirst errors:", errors[:5])

    raise RuntimeError(
        f"Preparation incomplete: "
        f"{len(errors)} errors, "
        f"{len(missing_prepared)} missing images."
    )

# ------------------------------------------------------------
# 6. Rebuild train, validation and test DataFrames
# ------------------------------------------------------------

train_df = prepared_df[
    prepared_df["split"] == "train"
].copy()

val_df = prepared_df[
    prepared_df["split"] == "validation"
].copy()

test_df = prepared_df[
    prepared_df["split"] == "test"
].copy()

assert len(train_df) == 15705
assert len(val_df) == 1746
assert len(test_df) == 7479

# ------------------------------------------------------------
# 7. Save Kaggle-specific image manifest
# ------------------------------------------------------------

manifest_path = (
    PROJECT_DIR / "data" /
    "prepared_manifest_kaggle.csv"
)

prepared_df.to_csv(
    manifest_path,
    index=False
)

elapsed = time.perf_counter() - start_time

# ------------------------------------------------------------
# 8. Final summary
# ------------------------------------------------------------

print("\n=== PREPARATION COMPLETED ===")

print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

print("New files:", created)
print("Reused files:", reused)
print("Missing files:", len(missing_prepared))

print(f"Duration: {elapsed / 60:.1f} minutes")
print("Manifest:", manifest_path)

print("\nSUCCESS: SRGAN images ready!")


In [ ]:

import tensorflow as tf
from pathlib import Path

# ============================================================
# RESTORE CUSTOM PIXELSHUFFLE LAYER
# AND LOAD TRAINED SRGAN GENERATOR
# ============================================================

@tf.keras.utils.register_keras_serializable(
    package="Midterm"
)
class PixelShuffle(tf.keras.layers.Layer):

    def __init__(self, scale=2, **kwargs):
        super().__init__(**kwargs)
        self.scale = int(scale)

    def call(self, inputs):
        return tf.nn.depth_to_space(
            inputs,
            block_size=self.scale
        )

    def get_config(self):
        config = super().get_config()
        config.update({
            "scale": self.scale
        })
        return config


# ------------------------------------------------------------
# 1. Locate saved generator
# ------------------------------------------------------------

PROJECT_DIR = Path("/kaggle/working/project")

generator_path = (
    PROJECT_DIR
    / "checkpoints"
    / "srgan"
    / "groupnorm_v2"
    / "best_generators"
    / "generator_best_psnr.keras"
)

assert generator_path.is_file(), (
    f"Generator file missing: {generator_path}"
)

# ------------------------------------------------------------
# 2. Load trained generator
# ------------------------------------------------------------

print("=== LOADING TRAINED SRGAN GENERATOR ===")
print("TensorFlow:", tf.__version__)

generator = tf.keras.models.load_model(
    str(generator_path),
    custom_objects={
        "PixelShuffle": PixelShuffle,
        "Midterm>PixelShuffle": PixelShuffle
    },
    compile=False
)

print("\nSUCCESS: Generator loaded!")

print("Model name:", generator.name)
print("Input shape:", generator.input_shape)
print("Output shape:", generator.output_shape)
print("Parameters:", generator.count_params())

# ------------------------------------------------------------
# 3. Test generator with a dummy image
# ------------------------------------------------------------

dummy_input = tf.random.uniform(
    shape=(1, 32, 32, 3),
    dtype=tf.float32
)

dummy_output = generator(
    dummy_input,
    training=False
)

assert tuple(dummy_output.shape) == (
    1, 128, 128, 3
), "Unexpected generator output shape!"

print("\n=== GENERATOR TEST ===")
print("Input:", dummy_input.shape)
print("Output:", dummy_output.shape)
print("Minimum output:", float(tf.reduce_min(dummy_output)))
print("Maximum output:", float(tf.reduce_max(dummy_output)))

print("\nSUCCESS: Generator compatibility test passed!")


In [ ]:

import tensorflow as tf
from tensorflow.keras import layers
from pathlib import Path

# ============================================================
# RESTORE COMPLETE SRGAN GROUPNORM V2 TRAINING STATE
# ============================================================

PROJECT_DIR = Path("/kaggle/working/project")

resume_dir_v2 = (
    PROJECT_DIR / "checkpoints" /
    "srgan" / "groupnorm_v2" / "resume"
)

print("=== SRGAN RECOVERY ===")
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

# ------------------------------------------------------------
# 1. Check previously loaded generator
# ------------------------------------------------------------

assert "generator" in globals(), (
    "First run the PixelShuffle generator loading cell."
)

assert generator.count_params() == 959427
assert tuple(generator.input_shape) == (None, 32, 32, 3)
assert tuple(generator.output_shape) == (None, 128, 128, 3)

print("\nGenerator architecture: OK")

# ------------------------------------------------------------
# 2. Rebuild GroupNorm discriminator
# ------------------------------------------------------------

def build_discriminator():

    inputs = layers.Input(shape=(128, 128, 3))

    # Eight convolutional blocks
    filters_and_strides = [
        (64, 1),
        (64, 2),
        (128, 1),
        (128, 2),
        (256, 1),
        (256, 2),
        (512, 1),
        (512, 2)
    ]

    x = inputs

    for i, (filters, stride) in enumerate(
        filters_and_strides
    ):

        x = layers.Conv2D(
            filters,
            kernel_size=3,
            strides=stride,
            padding="same"
        )(x)

        if i > 0:
            x = layers.GroupNormalization(
                groups=32,
                axis=-1,
                epsilon=1e-5
            )(x)

        x = layers.LeakyReLU(
            negative_slope=0.2
        )(x)

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(256)(x)

    x = layers.LeakyReLU(
        negative_slope=0.2
    )(x)

    outputs = layers.Dense(1)(x)

    return tf.keras.Model(
        inputs,
        outputs,
        name="SRGAN_Discriminator_GroupNorm"
    )


discriminator = build_discriminator()

print("Discriminator parameters:",
      discriminator.count_params())

assert discriminator.count_params() == 4820673, (
    "Discriminator architecture does not match "
    "the original GroupNorm v2."
)

print("Discriminator architecture: OK")

# ------------------------------------------------------------
# 3. Recreate original optimizers
# ------------------------------------------------------------

GENERATOR_LR = 1e-4
DISCRIMINATOR_LR = 1e-4

g_optimizer_v2 = tf.keras.optimizers.Adam(
    learning_rate=GENERATOR_LR,
    beta_1=0.9,
    global_clipnorm=1.0
)

d_optimizer_v2 = tf.keras.optimizers.Adam(
    learning_rate=DISCRIMINATOR_LR,
    beta_1=0.9,
    global_clipnorm=1.0
)

g_optimizer_v2.build(
    generator.trainable_variables
)

d_optimizer_v2.build(
    discriminator.trainable_variables
)

completed_gan_epochs_v2 = tf.Variable(
    0,
    dtype=tf.int64,
    trainable=False
)

# ------------------------------------------------------------
# 4. Construct complete training checkpoint
# ------------------------------------------------------------

checkpoint_v2 = tf.train.Checkpoint(
    generator=generator,
    discriminator=discriminator,
    generator_optimizer=g_optimizer_v2,
    discriminator_optimizer=d_optimizer_v2,
    epoch=completed_gan_epochs_v2
)

latest_checkpoint = tf.train.latest_checkpoint(
    str(resume_dir_v2)
)

if latest_checkpoint is None:
    raise FileNotFoundError(
        "No resume checkpoint found!"
    )

print("\nCheckpoint found:")
print(latest_checkpoint)

# ------------------------------------------------------------
# 5. Restore complete training state
# ------------------------------------------------------------

restore_status = checkpoint_v2.restore(
    latest_checkpoint
)

try:
    restore_status.assert_consumed()
    print("\nSUCCESS: Full checkpoint restored!")
    restore_complete = True

except (AssertionError, ValueError) as error:
    restore_complete = False
    print("\nWARNING: Full checkpoint verification failed.")
    print(str(error)[:3000])

# ------------------------------------------------------------
# 6. Verify restored training progress
# ------------------------------------------------------------

restored_epoch = int(
    completed_gan_epochs_v2.numpy()
)

g_iterations = int(
    g_optimizer_v2.iterations.numpy()
)

d_iterations = int(
    d_optimizer_v2.iterations.numpy()
)

print("\n=== RESTORED TRAINING STATE ===")
print("Completed epochs:", restored_epoch)
print("Next epoch:", restored_epoch + 1)
print("Generator parameters:", generator.count_params())
print("Discriminator parameters:",
      discriminator.count_params())
print("Generator optimizer iterations:", g_iterations)
print("Discriminator optimizer iterations:", d_iterations)
print("Complete restore:", restore_complete)

# Do not continue training unless restoration is verified
if (
    restore_complete
    and restored_epoch == 27
    and g_iterations == 27 * 982
    and d_iterations == 27 * 982
):
    print("\nSUCCESS: Ready to resume at epoch 28!")
else:
    print(
        "\nSTOP: Restore needs investigation "
        "before training."
    )


## Stage B — Continue epochs 28–125, checkpoint and upload
The historical cell originally had target 150, but that run was interrupted at epoch 125. It is configured here to stop at 125 deliberately. **Once the epoch-125 cloud upload finishes, refresh the Kaggle input checkpoint dataset** before the next section.

In [ ]:

# ============================================================
# SRGAN KAGGLE - UNATTENDED TRAINING WITH CLOUD BACKUPS
# Resume from epoch 27, continue toward epoch 150
# ============================================================

import os
import csv
import json
import time
import shutil
import tempfile
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import tensorflow as tf
import kagglehub

# ------------------------------------------------------------
# 1. Configuration and safety checks
# ------------------------------------------------------------

PROJECT_DIR = Path("/kaggle/working/project")

TOTAL_GAN_EPOCHS = 150
RUN_TO_EPOCH = 125

SR_BATCH_SIZE = 16
SEED = 42

BACKUP_INTERVAL = 5
ARCHIVE_INTERVAL = 10

BACKUP_DATASET = (
    "niklasohlenroth/applied-ai-midterm-srgan-checkpoints"
)

assert globals().get("restore_complete", False), (
    "STOP: Run the complete checkpoint restore cell first."
)

assert int(completed_gan_epochs_v2.numpy()) == 27, (
    "STOP: Expected completed epoch 27."
)

assert int(g_optimizer_v2.iterations.numpy()) == 27 * 982
assert int(d_optimizer_v2.iterations.numpy()) == 27 * 982

assert tf.config.list_physical_devices("GPU"), (
    "STOP: No GPU available."
)

assert len(train_df) == 15705
assert len(val_df) == 1746

print("=== SRGAN KAGGLE TRAINING ===")
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("Restored epoch:", int(completed_gan_epochs_v2.numpy()))
print("Training target:", RUN_TO_EPOCH)

# ------------------------------------------------------------
# 2. Rebuild original SRGAN input pipeline
# ------------------------------------------------------------

AUTOTUNE = tf.data.AUTOTUNE

def load_sr_pair(image_path, training=False):

    image_bytes = tf.io.read_file(image_path)

    hr = tf.image.decode_jpeg(
        image_bytes,
        channels=3
    )

    hr = tf.image.convert_image_dtype(
        hr,
        tf.float32
    )

    hr = tf.ensure_shape(
        hr,
        (128, 128, 3)
    )

    if training:
        hr = tf.image.random_flip_left_right(hr)

    lr = tf.image.resize(
        hr,
        (32, 32),
        method="bicubic",
        antialias=True
    )

    lr = tf.clip_by_value(lr, 0.0, 1.0)

    return lr, hr


def make_sr_dataset(df, training=False):

    ds = tf.data.Dataset.from_tensor_slices(
        df["prepared_path"].astype(str).tolist()
    )

    if training:
        ds = ds.shuffle(
            buffer_size=len(df),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        lambda path: load_sr_pair(
            path,
            training=training
        ),
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.batch(SR_BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)

    return ds


sr_train_ds = make_sr_dataset(
    train_df,
    training=True
)

sr_val_ds = make_sr_dataset(
    val_df,
    training=False
)

assert len(sr_train_ds) == 982
assert len(sr_val_ds) == 110

print("Training batches:", len(sr_train_ds))
print("Validation batches:", len(sr_val_ds))

# ------------------------------------------------------------
# 3. Restore pretrained VGG19 perceptual network
# ------------------------------------------------------------

print("\nLoading pretrained VGG19...")

vgg = tf.keras.applications.VGG19(
    include_top=False,
    weights="imagenet",
    input_shape=(128, 128, 3)
)

feature_extractor = tf.keras.Model(
    inputs=vgg.input,
    outputs=vgg.get_layer("block3_conv4").output
)

feature_extractor.trainable = False
del vgg

def perceptual_loss(hr_images, sr_images):

    hr_input = tf.keras.applications.vgg19.preprocess_input(
        hr_images * 255.0
    )

    sr_input = tf.keras.applications.vgg19.preprocess_input(
        sr_images * 255.0
    )

    hr_features = feature_extractor(
        hr_input,
        training=False
    )

    sr_features = feature_extractor(
        sr_input,
        training=False
    )

    return tf.reduce_mean(
        tf.square(hr_features - sr_features)
    )

# Original SRGAN v2 loss weights
PIXEL_WEIGHT = 1.0
PERCEPTUAL_WEIGHT = 1e-8
ADVERSARIAL_WEIGHT = 1e-3

bce_loss = tf.keras.losses.BinaryCrossentropy(
    from_logits=True
)

print("VGG19 ready.")

# ------------------------------------------------------------
# 4. Checkpoint and output directories
# ------------------------------------------------------------

gan_dir = (
    PROJECT_DIR / "checkpoints" /
    "srgan" / "groupnorm_v2"
)

resume_dir = gan_dir / "resume"
archive_dir = gan_dir / "archive"
best_dir = gan_dir / "best_generators"

archive_dir.mkdir(parents=True, exist_ok=True)
best_dir.mkdir(parents=True, exist_ok=True)

resume_manager = tf.train.CheckpointManager(
    checkpoint_v2,
    directory=str(resume_dir),
    max_to_keep=3
)

archive_manager = tf.train.CheckpointManager(
    checkpoint_v2,
    directory=str(archive_dir),
    max_to_keep=None
)

history_path = (
    PROJECT_DIR / "results" /
    "srgan_training_groupnorm_v2.csv"
)

tracking_path = (
    best_dir / "best_generator_tracking.json"
)

best_psnr_path = best_dir / "generator_best_psnr.keras"
best_ssim_path = best_dir / "generator_best_ssim.keras"

with open(tracking_path) as f:
    tracking = json.load(f)

best_psnr = float(tracking["best_psnr"]["value"])
best_ssim = float(tracking["best_ssim"]["value"])

# ------------------------------------------------------------
# 5. External backup to existing private Kaggle dataset
# ------------------------------------------------------------

backup_dir = Path(
    "/kaggle/working/srgan_cloud_backup"
)

def save_cloud_backup(epoch_number):

    # Build a portable project snapshot
    if backup_dir.exists():
        shutil.rmtree(backup_dir)

    snapshot = backup_dir / "project"

    snapshot_resume = (
        snapshot / "checkpoints" /
        "srgan/groupnorm_v2/resume"
    )

    snapshot_resume.mkdir(
        parents=True,
        exist_ok=True
    )

    latest = tf.train.latest_checkpoint(
        str(resume_dir)
    )

    if latest is None:
        raise RuntimeError(
            "Cannot upload backup: no checkpoint."
        )

    latest_name = Path(latest).name

    checkpoint_parts = [
        Path(latest + ".index"),
        *Path(latest).parent.glob(
            latest_name + ".data-*"
        )
    ]

    if len(checkpoint_parts) < 2:
        raise RuntimeError(
            "Incomplete checkpoint - backup aborted."
        )

    for part in checkpoint_parts:
        shutil.copy2(
            part,
            snapshot_resume / part.name
        )

    # Portable checkpoint pointer
    (snapshot_resume / "checkpoint").write_text(
        f'model_checkpoint_path: "{latest_name}"\n'
        f'all_model_checkpoint_paths: "{latest_name}"\n'
    )

    # Other essential project files
    important_files = [
        "data/dataset_split.csv",
        "results/srgan_training_groupnorm_v2.csv",
        "checkpoints/model_A/model_A_final.keras",
        "checkpoints/srgan/pretraining/generator_pretrain_best.keras",
        "documentation/srgan_architecture_v2.json",
        "documentation/srgan_training_config.json"
    ]

    for relative in important_files:

        source = PROJECT_DIR / relative

        if source.is_file():
            destination = snapshot / relative
            destination.parent.mkdir(
                parents=True,
                exist_ok=True
            )
            shutil.copy2(source, destination)

    # Best generators and tracking
    shutil.copytree(
        best_dir,
        snapshot / "checkpoints" /
        "srgan/groupnorm_v2/best_generators",
        dirs_exist_ok=True
    )

    # Recovery manifest
    manifest = {
        "platform": "Kaggle",
        "completed_epoch": epoch_number,
        "next_epoch": epoch_number + 1,
        "checkpoint": latest_name,
        "tensorflow_version": tf.__version__,
        "timestamp_utc": datetime.now(
            timezone.utc
        ).isoformat()
    }

    (snapshot / "recovery_manifest.json").write_text(
        json.dumps(manifest, indent=2)
    )

    print(
        f"\nUploading external backup: epoch {epoch_number}"
    )

    # Creates a new version of the user's existing dataset
    kagglehub.dataset_upload(
        BACKUP_DATASET,
        str(backup_dir),
        version_notes=(
            f"SRGAN Kaggle recovery after epoch {epoch_number}"
        )
    )

    print(
        f"EXTERNAL BACKUP UPLOADED: epoch {epoch_number}"
    )

# Test external backup BEFORE unattended training
current_epoch = int(
    completed_gan_epochs_v2.numpy()
)

print("\nTesting external checkpoint backup...")

save_cloud_backup(current_epoch)

print("\nExternal backup accepted. Training may begin.")

# ------------------------------------------------------------
# 6. Training and validation steps
# ------------------------------------------------------------

@tf.function
def srgan_train_step(lr_images, hr_images):

    # Train discriminator
    generated_for_d = tf.stop_gradient(
        generator(lr_images, training=False)
    )

    n = tf.shape(hr_images)[0]

    mixed_images = tf.concat(
        [hr_images, generated_for_d],
        axis=0
    )

    labels = tf.concat(
        [
            tf.ones((n, 1)),
            tf.zeros((n, 1))
        ],
        axis=0
    )

    indices = tf.random.shuffle(
        tf.range(tf.shape(mixed_images)[0])
    )

    mixed_images = tf.gather(
        mixed_images, indices
    )

    labels = tf.gather(
        labels, indices
    )

    with tf.GradientTape() as d_tape:

        logits = discriminator(
            mixed_images,
            training=True
        )

        d_loss = bce_loss(labels, logits)

    tf.debugging.assert_all_finite(
        d_loss,
        "Discriminator loss is not finite"
    )

    d_grads = d_tape.gradient(
        d_loss,
        discriminator.trainable_variables
    )

    d_optimizer_v2.apply_gradients(
        zip(
            d_grads,
            discriminator.trainable_variables
        )
    )

    # Train generator
    with tf.GradientTape() as g_tape:

        sr = generator(
            lr_images,
            training=True
        )

        fake_logits = discriminator(
            sr,
            training=False
        )

        pixel = tf.reduce_mean(
            tf.square(hr_images - sr)
        )

        feature = perceptual_loss(
            hr_images,
            sr
        )

        adversarial = bce_loss(
            tf.ones_like(fake_logits),
            fake_logits
        )

        weighted_pixel = PIXEL_WEIGHT * pixel
        weighted_feature = PERCEPTUAL_WEIGHT * feature
        weighted_adv = ADVERSARIAL_WEIGHT * adversarial

        g_loss = (
            weighted_pixel +
            weighted_feature +
            weighted_adv
        )

    tf.debugging.assert_all_finite(
        g_loss,
        "Generator loss is not finite"
    )

    g_grads = g_tape.gradient(
        g_loss,
        generator.trainable_variables
    )

    g_optimizer_v2.apply_gradients(
        zip(
            g_grads,
            generator.trainable_variables
        )
    )

    return (
        g_loss,
        d_loss,
        weighted_pixel,
        weighted_feature,
        weighted_adv
    )


@tf.function
def srgan_validate_step(lr_images, hr_images):

    sr = tf.clip_by_value(
        generator(lr_images, training=False),
        0.0,
        1.0
    )

    mse = tf.reduce_mean(
        tf.square(hr_images - sr),
        axis=[1, 2, 3]
    )

    psnr = tf.image.psnr(
        hr_images,
        sr,
        max_val=1.0
    )

    ssim = tf.image.ssim(
        hr_images,
        sr,
        max_val=1.0
    )

    return mse, psnr, ssim

# ------------------------------------------------------------
# 7. Atomic history saving
# ------------------------------------------------------------

history_columns = [
    "epoch",
    "generator_loss",
    "discriminator_loss",
    "pixel_loss",
    "perceptual_loss",
    "adversarial_loss",
    "val_mse",
    "val_psnr",
    "val_ssim",
    "duration_seconds",
    "timestamp_utc"
]

def write_history(row):

    with open(history_path, newline="") as f:
        existing_rows = list(csv.DictReader(f))

    existing_rows = [
        r for r in existing_rows
        if int(r["epoch"]) != row["epoch"]
    ]

    existing_rows.append(row)

    existing_rows.sort(
        key=lambda r: int(r["epoch"])
    )

    temp_path = history_path.with_suffix(".tmp")

    with open(temp_path, "w", newline="") as f:

        writer = csv.DictWriter(
            f,
            fieldnames=history_columns
        )

        writer.writeheader()
        writer.writerows(existing_rows)

    os.replace(temp_path, history_path)

# ------------------------------------------------------------
# 8. Automatic training loop
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("STARTING UNATTENDED SRGAN TRAINING")
print("Completed epochs:", current_epoch)
print("Target:", RUN_TO_EPOCH)
print("Backup every:", BACKUP_INTERVAL, "epochs")
print("=" * 55)

for epoch_number in range(
    current_epoch + 1,
    RUN_TO_EPOCH + 1
):

    start_time = time.perf_counter()

    g_metric = tf.keras.metrics.Mean()
    d_metric = tf.keras.metrics.Mean()

    pixel_metric = tf.keras.metrics.Mean()
    feature_metric = tf.keras.metrics.Mean()
    adv_metric = tf.keras.metrics.Mean()

    print(
        f"\nEpoch {epoch_number}/{TOTAL_GAN_EPOCHS}",
        flush=True
    )

    for step, (lr, hr) in enumerate(
        sr_train_ds,
        start=1
    ):

        g, d, pixel, feature, adv = srgan_train_step(
            lr, hr
        )

        g_metric.update_state(g)
        d_metric.update_state(d)
        pixel_metric.update_state(pixel)
        feature_metric.update_state(feature)
        adv_metric.update_state(adv)

        if step % 100 == 0 or step == len(sr_train_ds):

            print(
                f"Batch {step}/{len(sr_train_ds)} | "
                f"G: {float(g_metric.result()):.5f} | "
                f"D: {float(d_metric.result()):.5f}",
                flush=True
            )

    # Validation
    mse_total = 0.0
    psnr_total = 0.0
    ssim_total = 0.0
    count = 0

    for lr, hr in sr_val_ds:

        mse, psnr, ssim = srgan_validate_step(
            lr, hr
        )

        mse_total += float(tf.reduce_sum(mse))
        psnr_total += float(tf.reduce_sum(psnr))
        ssim_total += float(tf.reduce_sum(ssim))

        count += int(tf.shape(lr)[0])

    val_mse = mse_total / count
    val_psnr = psnr_total / count
    val_ssim = ssim_total / count

    elapsed = time.perf_counter() - start_time

    # Full checkpoint every completed epoch
    completed_gan_epochs_v2.assign(epoch_number)

    saved_checkpoint = resume_manager.save(
        checkpoint_number=epoch_number
    )

    if epoch_number % ARCHIVE_INTERVAL == 0:

        archive_manager.save(
            checkpoint_number=epoch_number
        )

    row = {
        "epoch": epoch_number,
        "generator_loss": float(g_metric.result()),
        "discriminator_loss": float(d_metric.result()),
        "pixel_loss": float(pixel_metric.result()),
        "perceptual_loss": float(feature_metric.result()),
        "adversarial_loss": float(adv_metric.result()),
        "val_mse": val_mse,
        "val_psnr": val_psnr,
        "val_ssim": val_ssim,
        "duration_seconds": round(elapsed, 2),
        "timestamp_utc": datetime.now(
            timezone.utc
        ).isoformat()
    }

    write_history(row)

    # Best generator exports
    if val_psnr > best_psnr:

        best_psnr = val_psnr
        generator.save(str(best_psnr_path))

        tracking["best_psnr"] = {
            "epoch": epoch_number,
            "value": val_psnr,
            "path": str(best_psnr_path)
        }

        print("New best PSNR generator saved.")

    if val_ssim > best_ssim:

        best_ssim = val_ssim
        generator.save(str(best_ssim_path))

        tracking["best_ssim"] = {
            "epoch": epoch_number,
            "value": val_ssim,
            "path": str(best_ssim_path)
        }

        print("New best SSIM generator saved.")

    tracking["latest_exported_epoch"] = epoch_number

    with open(tracking_path, "w") as f:
        json.dump(tracking, f, indent=2)

    print("\nEpoch summary")
    print("-" * 48)

    print(
        f"G: {row['generator_loss']:.6f} | "
        f"D: {row['discriminator_loss']:.6f}"
    )

    print(f"PSNR: {val_psnr:.3f} dB")
    print(f"SSIM: {val_ssim:.4f}")
    print(f"Duration: {elapsed:.1f} seconds")
    print("Checkpoint:", saved_checkpoint)

    # Persistent backup every 5 epochs
    if epoch_number % BACKUP_INTERVAL == 0:

        save_cloud_backup(epoch_number)

    print("-" * 48, flush=True)

print("\nTRAINING TARGET COMPLETED.")
print("Final epoch:", int(completed_gan_epochs_v2.numpy()))

# Final external backup
save_cloud_backup(
    int(completed_gan_epochs_v2.numpy())
)


## Stage C — New Kaggle session: restore epoch 125 and finish 126–150
**Run the following cells in a new Kaggle session with the uploaded epoch-125 dataset attached.** The assertions intentionally guard against restoring the wrong checkpoint.

In [ ]:

from pathlib import Path
from PIL import Image, ImageOps
import tensorflow as tf
import pandas as pd
import hashlib
import shutil
import os
import time

# ============================================================
# KAGGLE RECOVERY - EPOCH 125 + IMAGE PREPARATION
# ============================================================

BACKUP_ROOT = Path(
    "/kaggle/input/datasets/niklasohlenroth/"
    "applied-ai-midterm-srgan-checkpoints/project"
)

PROJECT_DIR = Path("/kaggle/working/project")

DOGS_DATASET_ROOT = Path(
    "/kaggle/input/datasets/blourdhuraju/"
    "dogs-vs-cats-classification/"
    "dogs-vs-cats-classification"
)

PREPARED_DIR = Path("/kaggle/working/prepared_rgb128")

# ------------------------------------------------------------
# 1. Restore latest cloud backup
# ------------------------------------------------------------

source_resume = (
    BACKUP_ROOT / "checkpoints/srgan/groupnorm_v2/resume"
)

source_ckpt = tf.train.latest_checkpoint(str(source_resume))

assert source_ckpt is not None
assert Path(source_ckpt).name == "ckpt-125", (
    f"Unexpected cloud checkpoint: {source_ckpt}"
)

working_resume = (
    PROJECT_DIR / "checkpoints/srgan/groupnorm_v2/resume"
)

working_ckpt = tf.train.latest_checkpoint(str(working_resume))

def checkpoint_epoch(path):
    return int(Path(path).name.split("-")[-1]) if path else -1

if checkpoint_epoch(working_ckpt) < 125:
    print("Restoring epoch 125 from cloud backup...")
    shutil.copytree(
        BACKUP_ROOT,
        PROJECT_DIR,
        dirs_exist_ok=True
    )
else:
    print("Existing working checkpoint is already current.")

latest = tf.train.latest_checkpoint(str(working_resume))

assert latest is not None
assert checkpoint_epoch(latest) >= 125

print("Restored checkpoint:", latest)

# ------------------------------------------------------------
# 2. Load original dataset split
# ------------------------------------------------------------

split_df = pd.read_csv(
    PROJECT_DIR / "data/dataset_split.csv"
)

assert len(split_df) == 24930

expected_counts = {
    "train": 15705,
    "validation": 1746,
    "test": 7479
}

assert split_df["split"].value_counts().to_dict() == expected_counts

prefix = "dogs-vs-cats-classification/"

def fix_path(path):
    path = str(path).replace("\\", "/")
    if path.startswith(prefix):
        path = path[len(prefix):]
    return path

split_df["image_path"] = (
    split_df["relative_path"]
    .apply(lambda p: str(DOGS_DATASET_ROOT / fix_path(p)))
)

assert split_df["image_path"].map(
    lambda p: Path(p).is_file()
).all()

print("\nOriginal images:", len(split_df))

# ------------------------------------------------------------
# 3. Prepare RGB images
# ------------------------------------------------------------

PREPARED_DIR.mkdir(parents=True, exist_ok=True)

def prepared_path(relative_path):
    name = hashlib.sha256(
        str(relative_path).encode("utf-8")
    ).hexdigest()[:24]
    return PREPARED_DIR / f"{name}.jpg"

prepared_df = split_df.copy()

prepared_df["prepared_path"] = (
    prepared_df["relative_path"]
    .apply(lambda p: str(prepared_path(p)))
)

created = 0
reused = 0
errors = []
start = time.perf_counter()

print("\n=== IMAGE PREPARATION ===")

for i, row in enumerate(prepared_df.itertuples(index=False), 1):

    src = Path(row.image_path)
    dst = Path(row.prepared_path)

    if dst.is_file() and dst.stat().st_size > 0:
        reused += 1
    else:
        tmp = dst.with_suffix(".part")

        try:
            with Image.open(src) as img:
                img = ImageOps.exif_transpose(img)
                img = img.convert("RGB")
                img = img.resize(
                    (128, 128),
                    Image.Resampling.BICUBIC
                )
                img.save(tmp, format="JPEG", quality=95)

            os.replace(tmp, dst)
            created += 1

        except Exception as e:
            errors.append((str(src), str(e)))
            tmp.unlink(missing_ok=True)

    if i % 2500 == 0 or i == len(prepared_df):
        print(
            f"{i:,}/{len(prepared_df):,} | "
            f"New: {created:,} | "
            f"Reused: {reused:,} | "
            f"Errors: {len(errors)}",
            flush=True
        )

assert not errors, f"Preparation errors: {errors[:5]}"

assert all(
    Path(p).is_file()
    for p in prepared_df["prepared_path"]
)

# ------------------------------------------------------------
# 4. Restore train / validation / test DataFrames
# ------------------------------------------------------------

train_df = prepared_df[
    prepared_df["split"] == "train"
].copy()

val_df = prepared_df[
    prepared_df["split"] == "validation"
].copy()

test_df = prepared_df[
    prepared_df["split"] == "test"
].copy()

assert len(train_df) == 15705
assert len(val_df) == 1746
assert len(test_df) == 7479

manifest_path = (
    PROJECT_DIR / "data/prepared_manifest_kaggle.csv"
)

prepared_df.to_csv(manifest_path, index=False)

print("\n=== RECOVERY COMPLETED ===")
print("Checkpoint:", latest)
print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))
print("New images:", created)
print("Reused images:", reused)
print("Errors:", len(errors))
print(f"Duration: {(time.perf_counter()-start)/60:.1f} min")
print("\nSUCCESS: Ready for model restoration!")


In [ ]:

import tensorflow as tf
from tensorflow.keras import layers
from pathlib import Path

# ============================================================
# KAGGLE - RESTORE COMPLETE SRGAN FROM EPOCH 125
# ============================================================

PROJECT_DIR = Path("/kaggle/working/project")

restore_complete = False

resume_dir_v2 = (
    PROJECT_DIR / "checkpoints" /
    "srgan" / "groupnorm_v2" / "resume"
)

latest_checkpoint = tf.train.latest_checkpoint(
    str(resume_dir_v2)
)

print("=== SRGAN EPOCH 125 RECOVERY ===")
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("Checkpoint:", latest_checkpoint)

assert latest_checkpoint is not None
assert Path(latest_checkpoint).name == "ckpt-125", (
    "STOP: Expected checkpoint 125."
)

assert len(train_df) == 15705
assert len(val_df) == 1746
assert len(test_df) == 7479

# ------------------------------------------------------------
# 1. Register custom PixelShuffle layer
# ------------------------------------------------------------

@tf.keras.utils.register_keras_serializable(
    package="Midterm"
)
class PixelShuffle(tf.keras.layers.Layer):

    def __init__(self, scale=2, **kwargs):
        super().__init__(**kwargs)
        self.scale = int(scale)

    def call(self, inputs):
        return tf.nn.depth_to_space(
            inputs,
            block_size=self.scale
        )

    def get_config(self):
        config = super().get_config()
        config["scale"] = self.scale
        return config

# ------------------------------------------------------------
# 2. Load generator architecture
# ------------------------------------------------------------

generator_path = (
    PROJECT_DIR / "checkpoints" /
    "srgan" / "groupnorm_v2" /
    "best_generators" /
    "generator_best_psnr.keras"
)

assert generator_path.is_file()

generator = tf.keras.models.load_model(
    str(generator_path),
    custom_objects={
        "PixelShuffle": PixelShuffle,
        "Midterm>PixelShuffle": PixelShuffle
    },
    compile=False
)

assert generator.count_params() == 959427

print("\nGenerator architecture restored.")
print("Parameters:", generator.count_params())

# ------------------------------------------------------------
# 3. Rebuild GroupNorm discriminator
# ------------------------------------------------------------

def build_discriminator():

    inputs = layers.Input(shape=(128, 128, 3))

    filters_and_strides = [
        (64, 1),
        (64, 2),
        (128, 1),
        (128, 2),
        (256, 1),
        (256, 2),
        (512, 1),
        (512, 2)
    ]

    x = inputs

    for i, (filters, stride) in enumerate(
        filters_and_strides
    ):

        x = layers.Conv2D(
            filters,
            kernel_size=3,
            strides=stride,
            padding="same"
        )(x)

        if i > 0:
            x = layers.GroupNormalization(
                groups=32,
                axis=-1,
                epsilon=1e-5
            )(x)

        x = layers.LeakyReLU(
            negative_slope=0.2
        )(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256)(x)
    x = layers.LeakyReLU(negative_slope=0.2)(x)

    outputs = layers.Dense(1)(x)

    return tf.keras.Model(
        inputs,
        outputs,
        name="SRGAN_Discriminator_GroupNorm"
    )

discriminator = build_discriminator()

assert discriminator.count_params() == 4820673, (
    "Discriminator architecture mismatch!"
)

print("\nDiscriminator architecture restored.")
print("Parameters:", discriminator.count_params())

# ------------------------------------------------------------
# 4. Rebuild optimizers
# ------------------------------------------------------------

GENERATOR_LR = 1e-4
DISCRIMINATOR_LR = 1e-4

g_optimizer_v2 = tf.keras.optimizers.Adam(
    learning_rate=GENERATOR_LR,
    beta_1=0.9,
    global_clipnorm=1.0
)

d_optimizer_v2 = tf.keras.optimizers.Adam(
    learning_rate=DISCRIMINATOR_LR,
    beta_1=0.9,
    global_clipnorm=1.0
)

g_optimizer_v2.build(
    generator.trainable_variables
)

d_optimizer_v2.build(
    discriminator.trainable_variables
)

completed_gan_epochs_v2 = tf.Variable(
    0,
    dtype=tf.int64,
    trainable=False
)

# ------------------------------------------------------------
# 5. Restore complete checkpoint
# ------------------------------------------------------------

checkpoint_v2 = tf.train.Checkpoint(
    generator=generator,
    discriminator=discriminator,
    generator_optimizer=g_optimizer_v2,
    discriminator_optimizer=d_optimizer_v2,
    epoch=completed_gan_epochs_v2
)

print("\nRestoring complete training state...")

restore_status = checkpoint_v2.restore(
    latest_checkpoint
)

restore_status.assert_consumed()

# ------------------------------------------------------------
# 6. Verify restored training state
# ------------------------------------------------------------

restored_epoch = int(
    completed_gan_epochs_v2.numpy()
)

g_iterations = int(
    g_optimizer_v2.iterations.numpy()
)

d_iterations = int(
    d_optimizer_v2.iterations.numpy()
)

expected_iterations = 125 * 982

print("\n=== RESTORED TRAINING STATE ===")
print("Completed epochs:", restored_epoch)
print("Next epoch:", restored_epoch + 1)
print("Generator optimizer iterations:", g_iterations)
print("Discriminator optimizer iterations:", d_iterations)
print("Expected iterations:", expected_iterations)

assert restored_epoch == 125, (
    "Incorrect restored epoch!"
)

assert g_iterations == expected_iterations, (
    "Generator optimizer state mismatch!"
)

assert d_iterations == expected_iterations, (
    "Discriminator optimizer state mismatch!"
)

restore_complete = True

print("\nSUCCESS: FULL CHECKPOINT RESTORED!")
print("Ready to resume at epoch 126.")


In [ ]:

# ============================================================
# SRGAN GROUPNORM V2 - FINAL TRAINING BLOCK
# Resume from epoch 125 and train until epoch 150
# ============================================================

import os
import csv
import json
import time
import shutil
from pathlib import Path
from datetime import datetime, timezone

import tensorflow as tf
import pandas as pd
import kagglehub

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_DIR = Path("/kaggle/working/project")

TOTAL_GAN_EPOCHS = 150
RUN_TO_EPOCH = 150

SR_BATCH_SIZE = 16
SEED = 42

BACKUP_INTERVAL = 5
ARCHIVE_INTERVAL = 10

BACKUP_DATASET = (
    "niklasohlenroth/applied-ai-midterm-srgan-checkpoints"
)

tf.keras.utils.set_random_seed(SEED)

# ------------------------------------------------------------
# 2. Verify restored training environment
# ------------------------------------------------------------

assert globals().get("restore_complete", False), (
    "Run the epoch 125 restoration cell first."
)

assert "generator" in globals()
assert "discriminator" in globals()
assert "checkpoint_v2" in globals()
assert "g_optimizer_v2" in globals()
assert "d_optimizer_v2" in globals()

assert len(train_df) == 15705
assert len(val_df) == 1746

assert tf.config.list_physical_devices("GPU"), (
    "No GPU detected."
)

gan_dir = (
    PROJECT_DIR / "checkpoints" /
    "srgan" / "groupnorm_v2"
)

resume_dir = gan_dir / "resume"
archive_dir = gan_dir / "archive"
best_dir = gan_dir / "best_generators"

archive_dir.mkdir(parents=True, exist_ok=True)
best_dir.mkdir(parents=True, exist_ok=True)

history_path = (
    PROJECT_DIR / "results" /
    "srgan_training_groupnorm_v2.csv"
)

tracking_path = best_dir / "best_generator_tracking.json"

best_psnr_path = best_dir / "generator_best_psnr.keras"
best_ssim_path = best_dir / "generator_best_ssim.keras"

assert history_path.is_file()
assert tracking_path.is_file()
assert best_psnr_path.is_file()
assert best_ssim_path.is_file()

# Re-read the latest checkpoint.
# This also protects against restarting this cell after
# an interrupted, partially completed training epoch.

latest_checkpoint = tf.train.latest_checkpoint(
    str(resume_dir)
)

assert latest_checkpoint is not None

checkpoint_v2.restore(
    latest_checkpoint
).assert_consumed()

start_epoch = int(
    completed_gan_epochs_v2.numpy()
)

assert 125 <= start_epoch <= TOTAL_GAN_EPOCHS

print("=== FINAL SRGAN TRAINING ===")
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("Checkpoint:", latest_checkpoint)
print("Completed epochs:", start_epoch)
print("Target epoch:", RUN_TO_EPOCH)

# ------------------------------------------------------------
# 3. Restore training history and best models
# ------------------------------------------------------------

history_df = pd.read_csv(history_path)

last_history_epoch = int(
    history_df["epoch"].max()
)

assert last_history_epoch == start_epoch, (
    f"History/checkpoint mismatch: "
    f"history={last_history_epoch}, "
    f"checkpoint={start_epoch}"
)

with open(tracking_path) as f:
    tracking = json.load(f)

best_psnr = float(
    tracking["best_psnr"]["value"]
)

best_ssim = float(
    tracking["best_ssim"]["value"]
)

print("\nBest generator tracking:")
print("Best PSNR:", best_psnr)
print("Best SSIM:", best_ssim)

# ------------------------------------------------------------
# 4. Rebuild the original SRGAN input pipeline
# ------------------------------------------------------------

AUTOTUNE = tf.data.AUTOTUNE

def load_sr_pair(image_path, training=False):

    image_bytes = tf.io.read_file(image_path)

    hr = tf.image.decode_jpeg(
        image_bytes,
        channels=3
    )

    hr = tf.image.convert_image_dtype(
        hr,
        tf.float32
    )

    hr = tf.ensure_shape(
        hr,
        (128, 128, 3)
    )

    if training:
        hr = tf.image.random_flip_left_right(hr)

    lr = tf.image.resize(
        hr,
        (32, 32),
        method="bicubic",
        antialias=True
    )

    lr = tf.clip_by_value(
        lr, 0.0, 1.0
    )

    return lr, hr


def make_sr_dataset(df, training=False):

    paths = (
        df["prepared_path"]
        .astype(str)
        .tolist()
    )

    ds = tf.data.Dataset.from_tensor_slices(paths)

    if training:
        ds = ds.shuffle(
            buffer_size=len(paths),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        lambda path: load_sr_pair(
            path,
            training=training
        ),
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.batch(SR_BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)

    return ds


sr_train_ds = make_sr_dataset(
    train_df,
    training=True
)

sr_val_ds = make_sr_dataset(
    val_df,
    training=False
)

assert len(sr_train_ds) == 982
assert len(sr_val_ds) == 110

print("\nTraining batches:", len(sr_train_ds))
print("Validation batches:", len(sr_val_ds))

expected_iterations = start_epoch * len(sr_train_ds)

assert int(
    g_optimizer_v2.iterations.numpy()
) == expected_iterations

assert int(
    d_optimizer_v2.iterations.numpy()
) == expected_iterations

print("Optimizer states verified.")

# ------------------------------------------------------------
# 5. Restore VGG19 perceptual feature extractor
# ------------------------------------------------------------

print("\nLoading VGG19...")

vgg = tf.keras.applications.VGG19(
    include_top=False,
    weights="imagenet",
    input_shape=(128, 128, 3)
)

feature_extractor = tf.keras.Model(
    inputs=vgg.input,
    outputs=vgg.get_layer("block3_conv4").output
)

feature_extractor.trainable = False

del vgg

def perceptual_loss(hr_images, sr_images):

    hr_input = (
        tf.keras.applications.vgg19.preprocess_input(
            hr_images * 255.0
        )
    )

    sr_input = (
        tf.keras.applications.vgg19.preprocess_input(
            sr_images * 255.0
        )
    )

    hr_features = feature_extractor(
        hr_input,
        training=False
    )

    sr_features = feature_extractor(
        sr_input,
        training=False
    )

    return tf.reduce_mean(
        tf.square(hr_features - sr_features)
    )

PIXEL_WEIGHT = 1.0
PERCEPTUAL_WEIGHT = 1e-8
ADVERSARIAL_WEIGHT = 1e-3

bce_loss = tf.keras.losses.BinaryCrossentropy(
    from_logits=True
)

print("VGG19 ready.")

# ------------------------------------------------------------
# 6. Checkpoint managers
# ------------------------------------------------------------

resume_manager = tf.train.CheckpointManager(
    checkpoint_v2,
    directory=str(resume_dir),
    max_to_keep=3
)

archive_manager = tf.train.CheckpointManager(
    checkpoint_v2,
    directory=str(archive_dir),
    max_to_keep=None
)

# ------------------------------------------------------------
# 7. Save best generators safely
# ------------------------------------------------------------

def save_generator_atomic(model, destination):

    temporary = destination.with_name(
        destination.stem + "_staging.keras"
    )

    model.save(str(temporary))
    os.replace(temporary, destination)


def save_tracking_atomic(data):

    temporary = tracking_path.with_suffix(".tmp")

    with open(temporary, "w") as f:
        json.dump(data, f, indent=2)

    os.replace(temporary, tracking_path)

# ------------------------------------------------------------
# 8. External Kaggle backup
# ------------------------------------------------------------

backup_dir = Path(
    "/kaggle/working/srgan_cloud_backup"
)

def save_cloud_backup(epoch_number):

    print(
        f"\nPreparing external backup: epoch {epoch_number}",
        flush=True
    )

    if backup_dir.exists():
        shutil.rmtree(backup_dir)

    snapshot = backup_dir / "project"

    snapshot_resume = (
        snapshot / "checkpoints" /
        "srgan/groupnorm_v2/resume"
    )

    snapshot_resume.mkdir(
        parents=True,
        exist_ok=True
    )

    latest = tf.train.latest_checkpoint(
        str(resume_dir)
    )

    if latest is None:
        raise RuntimeError(
            "No checkpoint available for backup."
        )

    checkpoint_name = Path(latest).name

    assert checkpoint_name == f"ckpt-{epoch_number}"

    parts = [
        Path(latest + ".index"),
        *Path(latest).parent.glob(
            checkpoint_name + ".data-*"
        )
    ]

    assert len(parts) >= 2
    assert all(p.is_file() for p in parts)

    for part in parts:
        shutil.copy2(
            part,
            snapshot_resume / part.name
        )

    # Portable checkpoint pointer
    (snapshot_resume / "checkpoint").write_text(
        f'model_checkpoint_path: "{checkpoint_name}"\n'
        f'all_model_checkpoint_paths: "{checkpoint_name}"\n'
    )

    # Important project files
    important_files = [
        "data/dataset_split.csv",
        "results/srgan_training_groupnorm_v2.csv",
        "checkpoints/model_A/model_A_final.keras",
        "checkpoints/srgan/pretraining/generator_pretrain_best.keras",
        "documentation/srgan_architecture_v2.json",
        "documentation/srgan_training_config.json"
    ]

    for relative in important_files:

        source = PROJECT_DIR / relative

        if not source.is_file():
            raise FileNotFoundError(source)

        destination = snapshot / relative

        destination.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        shutil.copy2(
            source,
            destination
        )

    # Best generators and metadata
    shutil.copytree(
        best_dir,
        snapshot / "checkpoints" /
        "srgan/groupnorm_v2/best_generators",
        dirs_exist_ok=True
    )

    # Backup metadata
    manifest = {
        "platform": "Kaggle",
        "completed_epoch": epoch_number,
        "next_epoch": epoch_number + 1,
        "checkpoint": checkpoint_name,
        "tensorflow_version": tf.__version__,
        "timestamp_utc": datetime.now(
            timezone.utc
        ).isoformat()
    }

    (snapshot / "recovery_manifest.json").write_text(
        json.dumps(manifest, indent=2)
    )

    kagglehub.dataset_upload(
        BACKUP_DATASET,
        str(backup_dir),
        version_notes=(
            f"SRGAN recovery checkpoint epoch {epoch_number}"
        )
    )

    print(
        f"EXTERNAL BACKUP UPLOADED: epoch {epoch_number}",
        flush=True
    )
    print(
        "Dataset version may still be processing.",
        flush=True
    )

# ------------------------------------------------------------
# 9. SRGAN training step
# ------------------------------------------------------------

@tf.function
def train_step(lr_images, hr_images):

    # Discriminator update
    generated_for_d = tf.stop_gradient(
        generator(lr_images, training=False)
    )

    batch_size = tf.shape(hr_images)[0]

    combined_images = tf.concat(
        [hr_images, generated_for_d],
        axis=0
    )

    labels = tf.concat(
        [
            tf.ones((batch_size, 1)),
            tf.zeros((batch_size, 1))
        ],
        axis=0
    )

    indices = tf.random.shuffle(
        tf.range(tf.shape(combined_images)[0])
    )

    combined_images = tf.gather(
        combined_images, indices
    )

    labels = tf.gather(
        labels, indices
    )

    with tf.GradientTape() as d_tape:

        d_logits = discriminator(
            combined_images,
            training=True
        )

        d_loss = bce_loss(
            labels,
            d_logits
        )

    tf.debugging.assert_all_finite(
        d_loss,
        "Invalid discriminator loss"
    )

    d_grads = d_tape.gradient(
        d_loss,
        discriminator.trainable_variables
    )

    d_optimizer_v2.apply_gradients(
        zip(
            d_grads,
            discriminator.trainable_variables
        )
    )

    # Generator update
    with tf.GradientTape() as g_tape:

        sr_images = generator(
            lr_images,
            training=True
        )

        fake_logits = discriminator(
            sr_images,
            training=False
        )

        pixel_loss = tf.reduce_mean(
            tf.square(hr_images - sr_images)
        )

        feature_loss = perceptual_loss(
            hr_images,
            sr_images
        )

        adversarial_loss = bce_loss(
            tf.ones_like(fake_logits),
            fake_logits
        )

        weighted_pixel = (
            PIXEL_WEIGHT * pixel_loss
        )

        weighted_feature = (
            PERCEPTUAL_WEIGHT * feature_loss
        )

        weighted_adv = (
            ADVERSARIAL_WEIGHT * adversarial_loss
        )

        g_loss = (
            weighted_pixel
            + weighted_feature
            + weighted_adv
        )

    tf.debugging.assert_all_finite(
        g_loss,
        "Invalid generator loss"
    )

    g_grads = g_tape.gradient(
        g_loss,
        generator.trainable_variables
    )

    g_optimizer_v2.apply_gradients(
        zip(
            g_grads,
            generator.trainable_variables
        )
    )

    return (
        g_loss,
        d_loss,
        weighted_pixel,
        weighted_feature,
        weighted_adv
    )

# ------------------------------------------------------------
# 10. Validation step
# ------------------------------------------------------------

@tf.function
def validate_step(lr_images, hr_images):

    sr_images = generator(
        lr_images,
        training=False
    )

    sr_images = tf.clip_by_value(
        sr_images,
        0.0,
        1.0
    )

    mse = tf.reduce_mean(
        tf.square(hr_images - sr_images),
        axis=[1, 2, 3]
    )

    psnr = tf.image.psnr(
        hr_images,
        sr_images,
        max_val=1.0
    )

    ssim = tf.image.ssim(
        hr_images,
        sr_images,
        max_val=1.0
    )

    return mse, psnr, ssim

# ------------------------------------------------------------
# 11. Save training history
# ------------------------------------------------------------

history_columns = [
    "epoch",
    "generator_loss",
    "discriminator_loss",
    "pixel_loss",
    "perceptual_loss",
    "adversarial_loss",
    "val_mse",
    "val_psnr",
    "val_ssim",
    "duration_seconds",
    "timestamp_utc"
]

def write_history(row):

    with open(history_path, newline="") as f:
        rows = list(csv.DictReader(f))

    rows = [
        old for old in rows
        if int(old["epoch"]) != row["epoch"]
    ]

    rows.append(row)

    rows.sort(
        key=lambda item: int(item["epoch"])
    )

    temp_path = history_path.with_suffix(".tmp")

    with open(temp_path, "w", newline="") as f:

        writer = csv.DictWriter(
            f,
            fieldnames=history_columns
        )

        writer.writeheader()
        writer.writerows(rows)

    os.replace(temp_path, history_path)

# ------------------------------------------------------------
# 12. Final training loop
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("STARTING FINAL SRGAN TRAINING BLOCK")
print("Resume epoch:", start_epoch)
print("Target epoch:", RUN_TO_EPOCH)
print("Backup interval:", BACKUP_INTERVAL)
print("=" * 55)

for epoch_number in range(
    start_epoch + 1,
    RUN_TO_EPOCH + 1
):

    start_time = time.perf_counter()

    metrics = {
        "g": tf.keras.metrics.Mean(),
        "d": tf.keras.metrics.Mean(),
        "pixel": tf.keras.metrics.Mean(),
        "feature": tf.keras.metrics.Mean(),
        "adv": tf.keras.metrics.Mean()
    }

    print(
        f"\nEpoch {epoch_number}/{TOTAL_GAN_EPOCHS}",
        flush=True
    )

    # Training
    for step, (lr_images, hr_images) in enumerate(
        sr_train_ds,
        start=1
    ):

        g, d, pixel, feature, adv = train_step(
            lr_images,
            hr_images
        )

        metrics["g"].update_state(g)
        metrics["d"].update_state(d)

        metrics["pixel"].update_state(pixel)
        metrics["feature"].update_state(feature)
        metrics["adv"].update_state(adv)

        if step % 100 == 0 or step == len(sr_train_ds):

            print(
                f"Batch {step}/{len(sr_train_ds)} | "
                f"G: {float(metrics['g'].result()):.5f} | "
                f"D: {float(metrics['d'].result()):.5f}",
                flush=True
            )

    # Validation
    mse_total = 0.0
    psnr_total = 0.0
    ssim_total = 0.0
    count = 0

    for lr_images, hr_images in sr_val_ds:

        mse, psnr, ssim = validate_step(
            lr_images,
            hr_images
        )

        mse_total += float(tf.reduce_sum(mse))
        psnr_total += float(tf.reduce_sum(psnr))
        ssim_total += float(tf.reduce_sum(ssim))

        count += int(tf.shape(lr_images)[0])

    val_mse = mse_total / count
    val_psnr = psnr_total / count
    val_ssim = ssim_total / count

    elapsed = time.perf_counter() - start_time

    # Save complete training state
    completed_gan_epochs_v2.assign(epoch_number)

    checkpoint_path = resume_manager.save(
        checkpoint_number=epoch_number
    )

    if epoch_number % ARCHIVE_INTERVAL == 0:

        archive_manager.save(
            checkpoint_number=epoch_number
        )

    row = {
        "epoch": epoch_number,
        "generator_loss": float(metrics["g"].result()),
        "discriminator_loss": float(metrics["d"].result()),
        "pixel_loss": float(metrics["pixel"].result()),
        "perceptual_loss": float(metrics["feature"].result()),
        "adversarial_loss": float(metrics["adv"].result()),
        "val_mse": val_mse,
        "val_psnr": val_psnr,
        "val_ssim": val_ssim,
        "duration_seconds": round(elapsed, 2),
        "timestamp_utc": datetime.now(
            timezone.utc
        ).isoformat()
    }

    write_history(row)

    # Best PSNR generator
    if val_psnr > best_psnr:

        best_psnr = val_psnr

        save_generator_atomic(
            generator,
            best_psnr_path
        )

        tracking["best_psnr"] = {
            "epoch": epoch_number,
            "value": float(val_psnr),
            "path": str(best_psnr_path)
        }

        print("NEW BEST PSNR GENERATOR SAVED")

    # Best SSIM generator
    if val_ssim > best_ssim:

        best_ssim = val_ssim

        save_generator_atomic(
            generator,
            best_ssim_path
        )

        tracking["best_ssim"] = {
            "epoch": epoch_number,
            "value": float(val_ssim),
            "path": str(best_ssim_path)
        }

        print("NEW BEST SSIM GENERATOR SAVED")

    tracking["latest_exported_epoch"] = epoch_number

    save_tracking_atomic(tracking)

    # Summary
    print("\nEpoch summary")
    print("-" * 50)

    print(
        f"Generator Loss: "
        f"{row['generator_loss']:.6f}"
    )

    print(
        f"Discriminator Loss: "
        f"{row['discriminator_loss']:.6f}"
    )

    print(f"Validation PSNR: {val_psnr:.3f} dB")
    print(f"Validation SSIM: {val_ssim:.4f}")
    print(f"Duration: {elapsed:.1f} seconds")
    print("Checkpoint:", checkpoint_path)

    # External backup every five epochs
    if epoch_number % BACKUP_INTERVAL == 0:

        save_cloud_backup(epoch_number)

    print("-" * 50, flush=True)

print("\n" + "=" * 55)
print("SRGAN TRAINING COMPLETED")
print("Final epoch:", int(completed_gan_epochs_v2.numpy()))
print("Best PSNR:", tracking["best_psnr"])
print("Best SSIM:", tracking["best_ssim"])
print("=" * 55)



## Final SRGAN diagnostics
For the final reports and Classifier B, refresh Kaggle input to the completed epoch-150 checkpoint dataset. A representative training history is also preserved in the results and final scientific report.

In [ ]:

from pathlib import Path
import json
import pandas as pd
import tensorflow as tf

# ==========================================
# SRGAN FINAL BACKUP VERIFICATION
# ==========================================

BACKUP_ROOT = Path(
    "/kaggle/input/datasets/niklasohlenroth/"
    "applied-ai-midterm-srgan-checkpoints/project"
)

manifest_path = BACKUP_ROOT / "recovery_manifest.json"
resume_dir = (
    BACKUP_ROOT / "checkpoints/srgan/groupnorm_v2/resume"
)
history_path = (
    BACKUP_ROOT / "results/srgan_training_groupnorm_v2.csv"
)
tracking_path = (
    BACKUP_ROOT / "checkpoints/srgan/groupnorm_v2/"
    "best_generators/best_generator_tracking.json"
)

print("=== SRGAN FINAL RECOVERY CHECK ===")

if not BACKUP_ROOT.exists():
    raise FileNotFoundError(
        "Checkpoint dataset not attached!"
    )

# Check cloud backup
with open(manifest_path) as f:
    manifest = json.load(f)

print("\nCloud backup epoch:",
      manifest["completed_epoch"])

# Check full checkpoint
latest = tf.train.latest_checkpoint(str(resume_dir))

print("Latest checkpoint:", latest)

if latest:
    index_ok = Path(latest + ".index").is_file()
    data_files = list(
        resume_dir.glob(Path(latest).name + ".data-*")
    )
    print("Checkpoint index:", index_ok)
    print("Checkpoint data files:", len(data_files))

# Check training history
history = pd.read_csv(history_path)

print("\nHistory last epoch:",
      int(history["epoch"].max()))
print("Total logged epochs:", len(history))

print("\nLast 5 epochs:")
print(
    history[
        ["epoch", "generator_loss",
         "discriminator_loss", "val_psnr", "val_ssim"]
    ].tail(5).to_string(index=False)
)

# Check best generators
with open(tracking_path) as f:
    tracking = json.load(f)

print("\n=== BEST GENERATORS ===")
print("Best PSNR:", tracking["best_psnr"])
print("Best SSIM:", tracking["best_ssim"])

if (
    manifest["completed_epoch"] == 150
    and latest is not None
    and Path(latest).name == "ckpt-150"
    and index_ok
    and len(data_files) >= 1
    and int(history["epoch"].max()) == 150
):
    print("\nSUCCESS: SRGAN EPOCH 150 BACKUP VERIFIED!")
else:
    print("\nWARNING: Check latest Kaggle dataset version.")



In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path

# ==============================================
# SRGAN - FINAL TRAINING ANALYSIS
# ==============================================

BACKUP_ROOT = Path(
    "/kaggle/input/datasets/niklasohlenroth/"
    "applied-ai-midterm-srgan-checkpoints/project"
)

FIGURE_DIR = Path(
    "/kaggle/working/srgan_final_figures"
)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

history = pd.read_csv(
    BACKUP_ROOT / "results/srgan_training_groupnorm_v2.csv"
).sort_values("epoch")

assert len(history) == 150
assert history["epoch"].tolist() == list(range(1, 151))

# Best validation results
best_psnr_row = history.loc[
    history["val_psnr"].idxmax()
]

best_ssim_row = history.loc[
    history["val_ssim"].idxmax()
]

print("=== FINAL SRGAN RESULTS ===")
print("Total epochs:", len(history))

print(
    f"Best PSNR: {best_psnr_row['val_psnr']:.4f} dB "
    f"(epoch {int(best_psnr_row['epoch'])})"
)

print(
    f"Best SSIM: {best_ssim_row['val_ssim']:.4f} "
    f"(epoch {int(best_ssim_row['epoch'])})"
)

print("\nFinal epoch:")
print(
    history.tail(1)[
        ["epoch", "generator_loss",
         "discriminator_loss", "val_psnr", "val_ssim"]
    ].to_string(index=False)
)

# Confirm the selected generator files exist
best_dir = (
    BACKUP_ROOT / "checkpoints/srgan/"
    "groupnorm_v2/best_generators"
)

for filename in [
    "generator_best_psnr.keras",
    "generator_best_ssim.keras"
]:
    path = best_dir / filename
    assert path.is_file(), f"Missing: {path}"
    print("Verified model:", filename)

# ==============================================
# Figure 1 - Validation PSNR
# ==============================================

plt.figure(figsize=(10, 5))

plt.plot(
    history["epoch"],
    history["val_psnr"],
    label="SRGAN validation PSNR"
)

plt.axhline(
    25.689,
    linestyle="--",
    label="Pretrained generator"
)

plt.axhline(
    25.251,
    linestyle=":",
    label="Bicubic baseline"
)

plt.scatter(
    [best_psnr_row["epoch"]],
    [best_psnr_row["val_psnr"]],
    s=75,
    zorder=5,
    label="Best epoch"
)

plt.xlabel("Adversarial training epoch")
plt.ylabel("PSNR (dB)")
plt.title("SRGAN Validation PSNR")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "srgan_final_psnr.png",
    dpi=250
)
plt.savefig(
    FIGURE_DIR / "srgan_final_psnr.pdf"
)
plt.show()

# ==============================================
# Figure 2 - Validation SSIM
# ==============================================

plt.figure(figsize=(10, 5))

plt.plot(
    history["epoch"],
    history["val_ssim"],
    label="SRGAN validation SSIM"
)

plt.axhline(
    0.7436,
    linestyle="--",
    label="Pretrained generator"
)

plt.axhline(
    0.7164,
    linestyle=":",
    label="Bicubic baseline"
)

plt.scatter(
    [best_ssim_row["epoch"]],
    [best_ssim_row["val_ssim"]],
    s=75,
    zorder=5,
    label="Best epoch"
)

plt.xlabel("Adversarial training epoch")
plt.ylabel("SSIM")
plt.title("SRGAN Validation SSIM")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "srgan_final_ssim.png",
    dpi=250
)
plt.savefig(
    FIGURE_DIR / "srgan_final_ssim.pdf"
)
plt.show()

# ==============================================
# Figure 3 - GAN Losses
# ==============================================

plt.figure(figsize=(10, 5))

plt.plot(
    history["epoch"],
    history["generator_loss"],
    label="Generator loss"
)

plt.plot(
    history["epoch"],
    history["discriminator_loss"],
    label="Discriminator loss"
)

plt.yscale("log")
plt.xlabel("Adversarial training epoch")
plt.ylabel("Loss (log scale)")
plt.title("SRGAN Generator and Discriminator Losses")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "srgan_final_losses.png",
    dpi=250
)
plt.savefig(
    FIGURE_DIR / "srgan_final_losses.pdf"
)
plt.show()

print("\nSUCCESS: Final training figures created.")
print("Saved to:", FIGURE_DIR)
